# Session 5 · Prompting in Apps: Spreadsheets, Personal Style and Comparisons

Generative AI › Track 1 · Prompt Engineering

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-05_clean_sheet.py`

In [1]:
"""An assistant was asked: 'Clean this messy attendance sheet and summarise it.'
Below is the kind of pandas code it suggests (the same steps map to Google Sheets functions)."""
import pandas as pd

raw = pd.DataFrame({
    "Name": [" ali khan", "SARA AHMED", "Bilal  Raza", "sara ahmed", "Hina Shah", None],
    "Class": ["BSCS-1", "bscs-1", "BSCS-2", "BSCS-1", "BSCS-2", "BSCS-1"],
    "Attended": ["14", "16", "9", "16", "n/a", "12"],
    "Total": [16, 16, 16, 16, 16, 16],
})
df = raw.dropna(subset=["Name"]).copy()                       # = FILTER out blanks
df["Name"] = df["Name"].str.split().str.join(" ").str.title()  # = PROPER(TRIM())
df["Class"] = df["Class"].str.upper()                         # = UPPER()
df = df.drop_duplicates(subset=["Name", "Class"])              # = Data > Remove duplicates
df["Attended"] = pd.to_numeric(df["Attended"], errors="coerce").fillna(0).astype(int)
df["Attendance %"] = (df["Attended"] / df["Total"] * 100).round(1)
df["Status"] = df["Attendance %"].apply(lambda p: "OK" if p >= 75 else "At risk")  # = IF()
print(df.to_string(index=False))
print("\nClass averages:")
print(df.groupby("Class")["Attendance %"].mean().round(1).to_string())

      Name  Class  Attended  Total  Attendance %  Status
  Ali Khan BSCS-1        14     16          87.5      OK
Sara Ahmed BSCS-1        16     16         100.0      OK
Bilal Raza BSCS-2         9     16          56.2 At risk
 Hina Shah BSCS-2         0     16           0.0 At risk

Class averages:
Class
BSCS-1    93.8
BSCS-2    28.1


### Example · `session-05_style_profile.py`

In [2]:
"""'Train' an assistant on your style: extract a style profile from your own writing samples
and paste it into custom instructions."""
import re, statistics
samples = [
    "Quick update: the lab is moved to Room 4. Bring your laptops. Thanks!",
    "Reminder - quiz on Friday. Chapters 3 and 4 only. Good luck, you've got this!",
    "Assignment deadline extended to Monday. Please submit on the portal. Thanks!",
]
sentences = [s for t in samples for s in re.split(r"(?<=[.!?])\s+", t) if s]
words = [w for t in samples for w in re.findall(r"[A-Za-z']+", t)]
profile = {
    "avg_sentence_words": round(statistics.mean(len(s.split()) for s in sentences), 1),
    "exclamations_per_msg": round(sum(t.count("!") for t in samples) / len(samples), 2),
    "uses_contractions": any("'" in w for w in words),
    "common_signoff": max(set(t.split()[-1] for t in samples), key=lambda x: sum(t.endswith(x) for t in samples)),
}
print("Style profile:", profile)
print("\nCustom-instruction prompt:\n"
      f"Write like me: short sentences (~{profile['avg_sentence_words']} words), friendly and direct, "
      f"about {profile['exclamations_per_msg']} exclamation per message, "
      f"{'use' if profile['uses_contractions'] else 'avoid'} contractions, end with '{profile['common_signoff']}'.")

Style profile: {'avg_sentence_words': 4.3, 'exclamations_per_msg': 1.0, 'uses_contractions': True, 'common_signoff': 'Thanks!'}

Custom-instruction prompt:
Write like me: short sentences (~4.3 words), friendly and direct, about 1.0 exclamation per message, use contractions, end with 'Thanks!'.
